# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR^2 dataset on human mast cell-extracellular vesicle proteomics, loaded via its Croissant schema using the `mlcroissant` library. All objects in the dataset are referenced by their `@id` as per the Croissant specification.

### Dataset Source
The dataset source is provided via its Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available `recordSet`s and their constituent fields (all referenced by their `@id`).

A record set corresponds to a table-like resource, typically referencing one or more data files. Each record set has fields (columns).

In [ ]:
# List out RecordSets and their fields using @id
record_sets = list(metadata.recordSets)
print("Available record sets and their fields (by @id):\n")
for rs in record_sets:
    print(f"  RecordSet: {rs['@id']}")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    Field: {field['@id']} (label: {getattr(field, 'label', getattr(field, 'name', ''))})")
    else:
        print("    (No fields found)")

## 3. Data Extraction
Load data from one or more record sets into DataFrame objects. Use the RecordSet and Field `@id`s you inspected above.

The following demonstrates loading all records from each record set into a pandas DataFrame. Adjust `record_set_ids` as needed if the dataset schema is updated.

In [ ]:
# -- Inspect record set @ids and select for analysis --
record_set_ids = [rs['@id'] for rs in record_sets]
print("RecordSet @ids in the dataset:", record_set_ids)

# -- Load all records for each record set --
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")

# For demonstration, pick the largest record set for EDA. You may adjust the ID as needed.
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in {main_rs_id}:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on a numeric field
- Normalizing values
- Grouping by a categorical field

Please adapt the exact field `@id` depending on the field list above.

In [ ]:
# -- EDA: Choose field @ids for numeric analysis and grouping --
# Replace these with actual field @ids from section 2, e.g.:
# numeric_field_id = 'cr:PeptideCount'      # Example
# group_field_id = 'cr:SampleType'         # Example
numeric_field_id = None
group_field_id = None

# Auto-select a numeric column if possible
df = dataframes[main_rs_id]
numeric_field_id = None
for field in df.columns:
    if df[field].dtype.kind in 'if' and ('count' in field.lower() or 'abundance' in field.lower() or 'int' in field.lower() or 'float' in field.lower()):
        numeric_field_id = field
        break
if not numeric_field_id and len(df.select_dtypes(include='number').columns) > 0:
    numeric_field_id = df.select_dtypes(include='number').columns[0]

print(f"Automatically selected numeric field: {numeric_field_id}")

# Select a group field by presence of a few unique values
for field in df.columns:
    if df[field].nunique() < 15 and df[field].dtype == object:
        group_field_id = field
        break
print(f"Automatically selected group-by field: {group_field_id}")

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'if' else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (total: {len(filtered_df)})")

    # Normalize
    col_norm = numeric_field_id + "_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by another field if appropriate
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("No suitable numeric field was found for EDA.")

## 5. Visualization
Visualize a distribution or relationship between fields (e.g., histogram or boxplot for the numeric analysis, bar plot grouped by a field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric distribution
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped by a field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for plotting.")

## 6. Conclusion

In this notebook, you successfully loaded the FAIR^2 proteomics dataset using the `mlcroissant` library, reviewed its schema and record sets, and performed exploratory analysis and basic visualizations. All dataset elements were referenced consistently by their `@id` fields, useful for transparent, reproducible scientific workflows.

You can extend this notebook to perform further bioinformatics or machine-learning analyses, leveraging the rich metadata and structure provided by the Croissant schema.